## RCoxNet — Kaplan–Meier survival curves (Figure 2)

Reproduces paper Figure 2: pooled prognostic-index KM curves for all four cancer types.

**Prerequisites**
- Run `pipeline_rcoxnet.ipynb` Steps 1–7 first to load data and build the PPI network
  (the variables `clinical`, `W`, `nodes`, `node_to_idx`, `seed_by_patient`, `all_idx` must be in memory).
- Or run this notebook stand-alone: set `CANCER` below and the imports cell will reload everything.

Change `CANCER` in the imports cell to `BRCA` / `GBM` / `LUNG` / `OV`.

In [ ]:
import os, json, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')
%matplotlib inline

from rcoxnet.config import CANCER_CONFIGS, CLIN_FEATURES
from rcoxnet.data_loader import load_raw_data
from rcoxnet.mutations import parse_mutations
from rcoxnet.clinical import extract_clinical
from rcoxnet.network import build_transition_matrix
from rcoxnet.rwr import compute_rwr_scores
from rcoxnet.features import make_feature_df, select_top_k
from rcoxnet.model import CoxPhRWRNet, cox_loss, c_index
from rcoxnet.preprocessing import (
    censoring_time_balanced_split,
    normalise_genomic,
    normalise_clinical,
    prep,
)

CANCER = 'BRCA'   # change to LUNG / GBM / OV

ROOT   = os.getcwd()
cfg    = CANCER_CONFIGS[CANCER]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype  = torch.FloatTensor

print(f'Cancer : {CANCER}')
print(f'Device : {device}')

In [ ]:
# load data and build PPI network
mut_raw, clin_p_raw, clin_s_raw, ppi_path, _ = load_raw_data(ROOT, cfg)
seed_dict, mut_count_dict = parse_mutations(mut_raw)
clinical, seed_by_patient, _ = extract_clinical(
    clin_p_raw, clin_s_raw, seed_dict, mut_count_dict, cfg)

nodes, node_to_idx, W, _ = build_transition_matrix(ppi_path)
all_idx = np.arange(len(nodes))

print(f'Patients : {len(clinical)}  |  PPI genes : {len(nodes)}')

### Step 8 — Compute averaged risk scores

Prognostic index (PI) scores are averaged across all 20 test folds and saved to CSV. These can be used for Kaplan–Meier stratification or any downstream survival analysis.

In [ ]:
import copy

# Hyperparameters used to generate paper Figure 2 (from the AllPPI grid search).
# These differ from best_params.json (FullSearch), most notably K=300 for BRCA.
KM_PARAMS = {
    'BRCA': {'alpha': 0.8, 'K': 300, 'hidden1':  64, 'hidden2': 16, 'output_nodes':  4, 'lr': 1e-4, 'l2': 1e-2},
    'GBM':  {'alpha': 0.5, 'K': 800, 'hidden1':  32, 'hidden2': 16, 'output_nodes':  8, 'lr': 1e-4, 'l2': 1e-2},
    'LUNG': {'alpha': 0.8, 'K': 600, 'hidden1': 128, 'hidden2': 64, 'output_nodes': 16, 'lr': 1e-3, 'l2': 1e-2},
    'OV':   {'alpha': 0.5, 'K': 300, 'hidden1':  32, 'hidden2': 16, 'output_nodes':  8, 'lr': 3e-4, 'l2': 1e-2},
}
km_p = KM_PARAMS[CANCER]
EPOCHS_KM = 2000  # paper Figure 2 used 2000 epochs (ablation_km_allcancers.py)

# Re-run RWR with the AllPPI alpha (may differ from FullSearch alpha)
rwr_km = compute_rwr_scores(
    clinical, W, nodes, node_to_idx,
    seed_by_patient, feat_idx=all_idx, alpha=km_p['alpha'])
df_km    = make_feature_df(rwr_km, nodes, clinical)
X_km     = df_km[nodes].values.astype(np.float32)
yt_km    = df_km['OS_MONTHS'].values.astype(np.float32)
ye_km    = df_km['OS_STATUS'].values.astype(np.float32)
clin_km  = df_km[CLIN_FEATURES].values.astype(np.float32)

# Top-K gene selection with AllPPI K
X_km_filt, _, _, _ = select_top_k(X_km, nodes, yt_km, ye_km, k=km_p['K'])
print(f'KM feature matrix : {X_km_filt.shape}  (alpha={km_p["alpha"]}, K={km_p["K"]})')

km_model_kwargs = dict(
    input_nodes  = X_km_filt.shape[1],
    hidden_nodes1= km_p['hidden1'],
    hidden_nodes2= km_p['hidden2'],
    output_nodes = km_p['output_nodes'],
    n_clin       = len(CLIN_FEATURES),
)

pi_sum   = np.zeros(len(yt_km))
pi_count = np.zeros(len(yt_km), dtype=int)

for rep in range(N_REPEATS):
    rng = np.random.RandomState(rep * 7 + 13)
    tr, va, te = censoring_time_balanced_split(yt_km, ye_km, TEST_FRAC, VAL_FRAC, rng)

    Xg_tr, Xg_va, Xg_te = normalise_genomic(X_km_filt[tr], X_km_filt[va], X_km_filt[te])
    cl_tr, cl_va, cl_te  = normalise_clinical(clin_km[tr], clin_km[va], clin_km[te])

    x_tr, cl_tr_t, yt_tr, ye_tr = prep(device, dtype, Xg_tr, cl_tr, yt_km[tr], ye_km[tr])
    x_va, cl_va_t, yt_va, ye_va = prep(device, dtype, Xg_va, cl_va, yt_km[va], ye_km[va])
    x_te, cl_te_t, yt_te, ye_te = prep(device, dtype, Xg_te, cl_te, yt_km[te], ye_km[te])

    torch.manual_seed(rep)
    np.random.seed(rep)
    net = CoxPhRWRNet(**km_model_kwargs).to(device)
    opt = optim.Adam(net.parameters(), lr=km_p['lr'], weight_decay=km_p['l2'])
    # CosineAnnealingLR matches the training used to generate paper Figure 2
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_KM, eta_min=1e-6)
    best_val, best_state = 0.0, copy.deepcopy(net.state_dict())

    for epoch in range(EPOCHS_KM):
        net.train()
        opt.zero_grad()
        loss = cox_loss(net(x_tr, cl_tr_t), yt_tr, ye_tr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        opt.step()
        sch.step()
        if epoch % CHECKPOINT == 0:
            net.eval()
            with torch.no_grad():
                vc = c_index(net(x_va, cl_va_t), yt_va, ye_va).item()
            if vc > best_val:
                best_val  = vc
                best_state = copy.deepcopy(net.state_dict())

    net.load_state_dict(best_state)
    net.eval()

    # prep() sorts by descending time; undo the sort so scores map back to te indices
    sort_idx   = np.argsort(yt_km[te])[::-1]
    orig_order = np.argsort(sort_idx)
    with torch.no_grad():
        raw = net(x_te, cl_te_t).cpu().numpy().ravel()
    pi_sum[te]   += raw[orig_order]
    pi_count[te] += 1

valid = pi_count > 0
out   = pd.DataFrame({
    'OS_MONTHS':      yt_km[valid],
    'OS_STATUS':      ye_km[valid].astype(int),
    'avg_risk_score': pi_sum[valid] / pi_count[valid],
})

os.makedirs(os.path.join(ROOT, 'results', 'km_plots'), exist_ok=True)
out.to_csv(os.path.join(ROOT, 'results', 'km_plots', f'km_scores_{CANCER}_notebook.csv'), index=False)

print(f'Patients with test predictions : {valid.sum()}')
print(f'Saved → results/km_plots/km_scores_{CANCER}_notebook.csv')
out.head()

### Step 9 — Kaplan–Meier survival curves

Patients are split at the median prognostic index into high / low risk groups. Risk scores are scaled to [−1, 1] before computing the threshold (matches paper Figure 2).

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from matplotlib.lines import Line2D

# min-max scale to [-1, 1] — matches plot_km_1x4_screenstyle_scaled.py
scores  = out['avg_risk_score'].values
s_min, s_max = scores.min(), scores.max()
scaled  = 2 * (scores - s_min) / (s_max - s_min) - 1

thresh  = float(np.median(scaled))
high    = scaled >= thresh
low     = ~high

t_hi = out.loc[high, 'OS_MONTHS'].values
t_lo = out.loc[low,  'OS_MONTHS'].values
e_hi = out.loc[high, 'OS_STATUS'].values
e_lo = out.loc[low,  'OS_STATUS'].values

p = logrank_test(t_hi, t_lo, event_observed_A=e_hi, event_observed_B=e_lo).p_value
pval_str = f'{p:.5f}' if p >= 1e-5 else f'{p:.2e}'

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(4.5, 4))

kmf.fit(t_hi, e_hi)
kmf.plot_survival_function(ax=ax, ci_show=False, color='#C0392B', linewidth=1.4,
                           show_censors=True,
                           censor_styles={'marker': '+', 'ms': 4, 'mew': 0.8})
kmf.fit(t_lo, e_lo)
kmf.plot_survival_function(ax=ax, ci_show=False, color='#1565C0', linewidth=1.4,
                           show_censors=True,
                           censor_styles={'marker': '+', 'ms': 4, 'mew': 0.8})

ax.get_legend().remove()
handles = [
    Line2D([0], [0], color='#C0392B', linewidth=1.4, label=f'PI>={thresh:.3f}  (n={high.sum()})'),
    Line2D([0], [0], color='#1565C0', linewidth=1.4, label=f'PI<{thresh:.3f}   (n={low.sum()})'),
]
ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8)
ax.text(0.97, 0.55, f'p-value: {pval_str}', transform=ax.transAxes,
        fontsize=8, va='top', ha='right', color='#222222')

ax.set_title(CANCER, fontsize=11, fontweight='bold')
ax.set_xlabel('Overall Survival in Months', fontsize=9)
ax.set_ylabel('Survival Probability', fontsize=9)
ax.set_xlim(left=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'log-rank p = {p:.4e}')